## Lab 5
In this lab, you will implement the random forest algorithm to build models for regression.

In [18]:
import numpy as np
import pandas as pd
import seaborn as sbs
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

Import the data in 'ENB2012_data.csv', split it into training, testing and validation sets.

In [19]:
# Standar normalization
# x_norm = x - u / σ 

def standar_norm(X):
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)
    X_norm = (X - X_mean) / X_std
    return X_norm

def load_data(filename):
    df = pd.read_csv(filename)
    df = standar_norm(df)

    X = df.drop(columns=['Y1', 'Y2'])
    y1 = df[['Y1']]
    y2 = df[['Y2']]

    X_temp, X_test, y1_temp, y1_test, y2_temp, y2_test = train_test_split(
        X, y1, y2, test_size=0.2, random_state=42
    )

    X_train, X_val, y1_train, y1_val, y2_train, y2_val = train_test_split(
        X_temp, y1_temp, y2_temp, test_size=0.25, random_state=42 
    )

    return {
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y1_train': y1_train,
        'y1_val': y1_val,
        'y1_test': y1_test,
        'y2_train': y2_train,
        'y2_val': y2_val,
        'y2_test': y2_test,
    }

data = load_data('ENB2012_data.csv')

Built the function you need to train a normal decision tree:

In [20]:
def region_mse(y):
    return np.mean((y - np.mean(y))**2)

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ss_res / ss_tot

def best_split(X, y):
    m, n = X.shape
    best_feat, best_thresh, best_mse = None, None, float('inf')

    for feature in range(n):
        thresholds = np.unique(X[:, feature])
        for t in thresholds:
            left_mask = X[:, feature] <= t
            right_mask = X[:, feature] > t
            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue
            mse_left = mse(y[left_mask])
            mse_right = mse(y[right_mask])
            total_mse = (left_mask.sum() * mse_left + right_mask.sum() * mse_right) / m
            if total_mse < best_mse:
                best_feat, best_thresh, best_mse = feature, t, total_mse
    return best_feat, best_thresh

def build_tree(X, y, depth=0, max_depth=5, min_samples=5):
    if depth >= max_depth or len(y) <= min_samples:
        return np.mean(y) 

    feat, thresh = best_split(X, y)
    if feat is None:
        return np.mean(y)

    left_mask = X[:, feat] <= thresh
    right_mask = X[:, feat] > thresh

    left_branch = build_tree(X[left_mask], y[left_mask], depth + 1, max_depth, min_samples)
    right_branch = build_tree(X[right_mask], y[right_mask], depth + 1, max_depth, min_samples)

    return (feat, thresh, left_branch, right_branch)

def predict_one(x, tree):
    if not isinstance(tree, tuple):
        return tree
    feat, thresh, left, right = tree
    if x[feat] <= thresh:
        return predict_one(x, left)
    else:
        return predict_one(x, right)

def predict(X, tree):
    return np.array([predict_one(x, tree) for x in X])

Buit a function that train a random forest:

In [21]:
def bootstrap_sample(X, y):
    n = len(y)
    idxs = np.random.choice(n, size=n, replace=True)
    return X[idxs], y[idxs]

def best_split(X, y, feature_idxs):
    m, n = X.shape
    best_feat, best_thresh, best_mse = None, None, float('inf')

    for feature in feature_idxs:
        thresholds = np.unique(X[:, feature])
        for t in thresholds:
            left = X[:, feature] <= t
            right = X[:, feature] > t
            if left.sum() == 0 or right.sum() == 0:
                continue
            mse_left = region_mse(y[left])
            mse_right = region_mse(y[right])
            total_mse = (left.sum() * mse_left + right.sum() * mse_right) / m
            if total_mse < best_mse:
                best_feat, best_thresh, best_mse = feature, t, total_mse
    return best_feat, best_thresh

def build_tree(X, y, depth=0, max_depth=5, min_samples=5, max_features=None):
    if depth >= max_depth or len(y) <= min_samples:
        return np.mean(y)

    n_features = X.shape[1]
    feature_idxs = np.random.choice(n_features, size=max_features or n_features, replace=False)
    feat, thresh = best_split(X, y, feature_idxs)

    if feat is None:
        return np.mean(y)

    left = X[:, feat] <= thresh
    right = X[:, feat] > thresh

    left_branch = build_tree(X[left], y[left], depth + 1, max_depth, min_samples, max_features)
    right_branch = build_tree(X[right], y[right], depth + 1, max_depth, min_samples, max_features)

    return (feat, thresh, left_branch, right_branch)

def predict_one(x, tree):
    if not isinstance(tree, tuple):
        return tree
    feat, thresh, left, right = tree
    return predict_one(x, left) if x[feat] <= thresh else predict_one(x, right)

def predict_tree(X, tree):
    return np.array([predict_one(x, tree) for x in X])

def fit_random_forest(X, y, n_trees=10, max_depth=5, min_samples=5, max_features=None):
    forest = []
    for _ in range(n_trees):
        X_sample, y_sample = bootstrap_sample(X, y)
        tree = build_tree(X_sample, y_sample, max_depth=max_depth, min_samples=min_samples, max_features=max_features)
        forest.append(tree)
    return forest

def predict_random_forest(X, forest):
    preds = np.array([predict_tree(X, tree) for tree in forest])
    return np.mean(preds, axis=0)

Built a function that predicts the electric loads of a new point given a trained random forest

In [ ]:
def predict_point(X_point, forest):
    return np.mean([predict_one(X_point, tree) for tree in forest])

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ss_res / ss_tot

def evaluate_metrics(y_true, y_pred):
    return {
        'MSE': mse(y_true, y_pred),
        'MAE': mae(y_true, y_pred),
        'R2': r2_score(y_true, y_pred)
    }


def grid_search(X_train, y_train, X_val, y_val, param_grid):
    #find best hyperparameters for Random Forest
    best_params = None
    best_score = float('inf')

    for n_trees in param_grid['n_trees']:
        for max_depth in param_grid['max_depth']:
            for max_features in param_grid['max_features']:
                forest = fit_random_forest(
                    X_train.to_numpy(), y_train.to_numpy().ravel(),
                    n_trees=n_trees,
                    max_depth=max_depth,
                    min_samples=5,
                    max_features=max_features
                )
                y_pred = predict_random_forest(X_val.to_numpy(), forest)
                score = mse(y_val.to_numpy().ravel(), y_pred)

                if score < best_score:
                    best_score = score
                    best_params = {
                        'n_trees': n_trees,
                        'max_depth': max_depth,
                        'max_features': max_features
                    }

    return best_params

Builld functions to evaluate your predictions, use this functions to find optimal hyperparameters using also our validation set.

In [23]:
def train_and_evaluate_final_model(data, y_key='y1'):
    # Combine training and validation sets
    X_final_train = pd.concat([data['X_train'], data['X_val']])
    y_final_train = pd.concat([data[f'{y_key}_train'], data[f'{y_key}_val']])
    
    # Find best hyperparameters using grid search
    param_grid = {
        'n_trees': [5, 10, 20],
        'max_depth': [3, 5, 7],
        'max_features': [2, 4, 6]
    }
    
    best_params = grid_search(
        data['X_train'], data[f'{y_key}_train'],
        data['X_val'], data[f'{y_key}_val'],
        param_grid
    )
    
    print("Mejores hiperparámetros:", best_params)
    
    # Train final model with best hyperparameters
    forest = fit_random_forest(
        X_final_train.to_numpy(), y_final_train.to_numpy().ravel(),
        n_trees=best_params['n_trees'],
        max_depth=best_params['max_depth'],
        max_features=best_params['max_features']
    )

    
    return forest

forest_y1 = train_and_evaluate_final_model(data, y_key='y1')
forest_y2 = train_and_evaluate_final_model(data, y_key='y2')

Mejores hiperparámetros: {'n_trees': 10, 'max_depth': 7, 'max_features': 6}
Mejores hiperparámetros: {'n_trees': 20, 'max_depth': 7, 'max_features': 4}


Train a final random forest using your optimal hyperparameters and both the training and validation sets. Predict for the datapoints in the testing set and evaluate your final predictions.

In [24]:
def final_predictions(data, forest, y_key='y1'):
    # Predict on the test set
    # Evaluate feature importance
    X_test = data['X_test'].to_numpy()
    y_test = data[f'{y_key}_test'].to_numpy().ravel()
    #y_pred = predict_random_forest(X_test, forest)
    y_pred = predict_random_forest(data['X_test'].to_numpy(), forest)

    print("Evaluación en test:")
    print("MSE:", mse(y_test, y_pred))
    print("MAE:", mae(y_test, y_pred))
    print("R²:", r2_score(y_test, y_pred))

forest_y1 = final_predictions(data, forest_y1, y_key='y1')
forest_y2 = final_predictions(data, forest_y2, y_key='y2')

Evaluación en test:
MSE: 0.002864455518502819
MAE: 0.040070434600687745
R²: 0.9972056795240131
Evaluación en test:
MSE: 0.03506072874234932
MAE: 0.12652840224967596
R²: 0.9657990137222013
